# 08 – Gradient boosting (Poisson) op alle data, getuned

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | De beste techniek uit de 2024-studie (`05e`, `06`) opnieuw tunen op alle data (2013–2026), en twee vragen beantwoorden: **helpt meer data echt?** en **helpen de nieuwe kenmerken** (`year`, `active_stations`, `snow_depth_cm`)? Het resultaat gaat naar de API. |
| **Input** | `data/model_table.parquet`, `models/2024/best_params.json` |
| **Output** | `models/metrics/hist_gradient_boosting.json`, `models/metrics/hgb_one_year.json`, `models/predictions/*.parquet`, `models/data_volume.json`, `models/best_params.json`, `deploy/citi_bike_demand/model.pkl` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Waarom dit model?
In de 2024-studie won `HistGradientBoostingRegressor` met Poisson-verlies van de baseline, het Poisson-GLM, de random forest en alle modellen van PyCaret, in elke fold (`06_model_selectie_2024`). Het is bovendien klein en snel en draait met alleen scikit-learn op de server. Met 15 keer meer data (± 3 miljoen zone-uren) is snelheid nog belangrijker: de histogram-variant deelt elke kolom in 255 bakjes en schaalt daardoor goed.

## 0. Imports en data

In [1]:
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

import citibike as cb

table = cb.load_model_table()
X_train, y_train, _, X_test, y_test = cb.train_test(table)
train_time = table.loc[table["split"] == "train", "time"].reset_index(drop=True)
folds = cb.backtest_folds(train_time)
ZONE = [cb.FEATURES.index("zone")]
params_2024 = json.loads((cb.MODEL_2024 / "best_params.json").read_text())["params"]
print(f"train {len(X_train):,} / test {len(X_test):,} zone-uren; hyperparameters uit 2024: {params_2024}")


def hgb(features=cb.FEATURES, **params):
    return HistGradientBoostingRegressor(loss="poisson", categorical_features=[features.index("zone")],
                                         random_state=cb.SEED, **params)

train 2,556,912 / test 262,080 zone-uren; hyperparameters uit 2024: {'l2_regularization': 0.015314, 'learning_rate': 0.03129, 'max_iter': 1056, 'max_leaf_nodes': 85, 'min_samples_leaf': 18}


## 1. Vertrekpunt: de hyperparameters uit 2024
Eerst trainen we het model op alle data met de waarden die in 2024 het best waren, en meten we het met dezelfde backtest-folds als de baseline (`07`).

In [2]:
cv_2024 = cb.cv_scores(hgb(**params_2024), X_train, y_train, cv=folds)
cv_2024

{'cv_mae': [27.7858, 38.5101, 29.6838],
 'cv_mae_mean': 31.9932,
 'cv_poisson_deviance': [13.4004, 16.6771, 10.0803],
 'cv_fit_seconds': 187.3}

## 2. Opnieuw tunen op alle data
Met 15 keer meer data kan het model complexere bomen gebruiken zonder te overfitten. We zoeken opnieuw, met de waarden uit 2024 in het midden van de zoekruimte. Elke combinatie wordt met de **3 backtest-folds** gemeten (trainen op het verleden, valideren op het jaar erna).

Een fit op ± 2,5 miljoen rijen duurt minuten, dus we beperken ons tot 10 combinaties. `HistGradientBoosting` gebruikt bij zoveel data automatisch **early stopping** (10% van de trainrijen als interne validatie): `max_iter` is dus een bovengrens, het model stopt zelf als extra bomen niet meer helpen.

In [3]:
search = RandomizedSearchCV(
    hgb(),
    {"learning_rate": loguniform(0.02, 0.15), "max_iter": randint(500, 2000),
     "max_leaf_nodes": randint(63, 255), "min_samples_leaf": randint(20, 400),
     "l2_regularization": loguniform(1e-3, 10)},
    n_iter=10, cv=folds, scoring="neg_mean_absolute_error", random_state=cb.SEED, n_jobs=1)
search.fit(X_train, y_train)
results = (pd.DataFrame(search.cv_results_["params"])
             .assign(cv_mae=-search.cv_results_["mean_test_score"], std_over_folds=search.cv_results_["std_test_score"],
                     fit_seconds=search.cv_results_["mean_fit_time"])
             .sort_values("cv_mae"))
results.round(3)

,l2_regularization,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
8,0.005,0.023,887,151,335,31.773,4.837,69.591
9,0.179,0.043,1869,115,359,31.808,4.879,105.407
2,0.022,0.027,630,212,328,31.833,4.782,60.336
5,0.053,0.036,1582,232,207,31.892,4.959,118.642
1,0.244,0.049,1738,137,107,31.915,5.029,102.580
6,0.029,0.050,1186,252,70,32.160,4.967,90.724
4,9.308,0.069,1837,84,272,32.162,4.977,90.478
3,7.579,0.107,1305,192,211,32.381,4.962,86.107
0,0.031,0.136,1630,134,208,32.502,4.747,85.880
7,0.034,0.145,1654,113,154,32.640,4.700,80.955


**Interpretatie:** alle tien de combinaties liggen dicht bij elkaar (backtest-MAE 31,8 tot 32,6), terwijl de spreiding over de folds ± 4,8 is. De beste delen hetzelfde patroon als in 2024: een **kleine learning rate** (0,02–0,04) en **grote bomen** (115–212 bladeren). Nieuw is dat ze ook **grote bladeren** willen (`min_samples_leaf` 330–360 in plaats van 18): met 15 keer meer data kan elk blad op veel meer uren steunen, wat de voorspellingen stabieler maakt. De slechtste combinaties hebben weer een grote learning rate (0,11–0,15).

## 3. Beste model: backtest en eindtest

In [4]:
candidates = {"hyperparameters 2024": params_2024,
              "opnieuw getuned": {k: (round(float(v), 6) if isinstance(v, float) else int(v)) for k, v in search.best_params_.items()}}
scores = {"hyperparameters 2024": cv_2024,
          "opnieuw getuned": cb.cv_scores(hgb(**candidates["opnieuw getuned"]), X_train, y_train, cv=folds)}
display(pd.DataFrame({k: {"cv_mae": v["cv_mae_mean"], **{f"fold {i}": m for i, m in enumerate(v["cv_mae"])}}
                      for k, v in scores.items()}).T)
chosen = min(scores, key=lambda k: scores[k]["cv_mae_mean"])
params = candidates[chosen]
model = hgb(**params).fit(X_train, y_train)
pred = model.predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "08_hgb_all_data", "model": f"HistGradientBoosting (Poisson), alle data, {chosen}",
           "params": params, **scores[chosen], **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("hist_gradient_boosting", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred))
print("Gekozen:", chosen, params)
pd.Series(test_metrics)

,cv_mae,fold 0,fold 1,fold 2
hyperparameters 2024,31.9932,27.7858,38.5101,29.6838
opnieuw getuned,31.7859,27.3669,38.4969,29.4940


Gekozen: opnieuw getuned {'l2_regularization': 0.004809, 'learning_rate': 0.022801, 'max_iter': 887, 'max_leaf_nodes': 151, 'min_samples_leaf': 335}


mae                 28.9411
rmse                53.6342
r2                   0.9309
poisson_deviance    10.6251
dtype: float64

**Interpretatie:** de opnieuw getunede instellingen zijn maar een fractie beter dan die uit 2024 (backtest 31,8 tegenover 32,0), maar wel in **alle drie** de folds. Op de testset: **MAE 28,9, R² 0,931**. Dat de winst van het hertunen zo klein is, is op zich een nuttig resultaat: de hyperparameters uit de 2024-studie waren al bijna goed voor 13 jaar data. De grote stappen zaten in de data en de kenmerken (zie §4 en §5).

Het verschil tussen de folds (fold 1, validatie oktober 2023 – september 2024: 38,5; de andere ± 28) komt door de groei: in dat jaar groeide het netwerk het snelst (zie `02b` §6), en elke fold leert alleen van de jaren ervoor.

## 4. Helpt meer data echt? (leercurve over jaren)
De kernvraag van deze uitbreiding. We trainen hetzelfde model (zelfde hyperparameters) op steeds meer jaren vóór de testperiode, en meten telkens op dezelfde testset (de laatste 12 maanden):
- **1 jaar**: vergelijkbaar met de 2024-studie (één jaar trainingsdata);
- 2, 3, 5 jaar, en **alles** (sinds 2013).

Meer data is niet vanzelf beter: het gedrag verandert (e-bikes, corona, nieuwe buurten), en oude jaren kunnen het model naar een verouderd patroon trekken.

In [5]:
test_start = table.loc[table["split"] == "test", "time"].min()
volume_rows = []
for years in [1, 2, 3, 5, None]:
    since = test_start - pd.DateOffset(years=years) if years else train_time.min()
    keep = (train_time >= since).to_numpy()
    m = hgb(**params).fit(X_train[keep], y_train[keep])
    p = m.predict(X_test)
    volume_rows.append({"trainingsdata": f"laatste {years} jaar" if years else "alles (sinds 2013)",
                        "zone-uren": int(keep.sum()), **cb.evaluate(y_test, p)})
    if years == 1:
        cb.save_results("hgb_one_year", {"notebook": "08_hgb_all_data", "model": "HistGradientBoosting, alleen het laatste jaar",
                                         "params": params, **{f"test_{k}": v for k, v in cb.evaluate(y_test, p).items()}},
                        table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=p))
volume = pd.DataFrame(volume_rows).set_index("trainingsdata")
(cb.MODEL_DIR / "data_volume.json").write_text(volume.reset_index().to_json(orient="records", indent=2))
volume

,zone-uren,mae,rmse,r2,poisson_deviance
trainingsdata,,,,,
laatste 1 jaar,262800,30.3095,55.9832,0.9248,12.5231
laatste 2 jaar,526320,29.7777,54.7821,0.9280,11.9484
laatste 3 jaar,781824,29.3374,53.9275,0.9302,11.5618
laatste 5 jaar,1278408,29.0976,53.7071,0.9308,11.1198
alles (sinds 2013),2556912,28.9411,53.6342,0.9309,10.6251


**Interpretatie:** **ja, meer data helpt**, en wel bij elke stap: 30,3 (1 jaar) → 29,8 (2 jaar) → 29,3 (3 jaar) → 29,1 (5 jaar) → **28,9 (alles)**. De winst neemt af naarmate er meer jaren bijkomen: de oudste jaren (2013–2020, een veel kleiner netwerk zonder e-bikes) voegen nog maar weinig toe, maar ze schaden ook niet. Dankzij `year` en `active_stations` kan het model het oude gedrag van het nieuwe onderscheiden in plaats van het te middelen.

## 5. Helpen de nieuwe kenmerken? (ablatie)
Ten opzichte van 2024 kwamen er drie kenmerken bij:
- `year` (trend) en `active_stations` (grootte van het netwerk in de zone), uit de EDA over alle jaren (`02b`);
- `snow_depth_cm` (sneeuw op de grond), een **iteratie na de foutenanalyse**: in een eerste versie van `09` zaten de grootste fouten op droge dagen ná een sneeuwstorm (26–31 januari 2026 tot 9 keer te veel voorspeld). Het model kende alleen de neerslag van het uur zelf en wist niet dat de straten nog wit lagen.

We laten ze één voor één weg en meten opnieuw, op de testset én op de backtest-folds.

In [6]:
ablation = {}
for name, drop in [("alle kenmerken", []), ("zonder year", ["year"]), ("zonder active_stations", ["active_stations"]),
                   ("zonder snow_depth_cm", ["snow_depth_cm"]),
                   ("zonder alle drie (= kenmerken uit 2024)", ["year", "active_stations", "snow_depth_cm"])]:
    feats = [f for f in cb.FEATURES if f not in drop]
    backtest = cb.cv_scores(hgb(feats, **params), X_train[feats], y_train, cv=folds)
    m = hgb(feats, **params).fit(X_train[feats], y_train)
    ablation[name] = {"backtest_mae": backtest["cv_mae_mean"], **{f"backtest fold {i}": v for i, v in enumerate(backtest["cv_mae"])},
                      **cb.evaluate(y_test, m.predict(X_test[feats]))}
pd.DataFrame(ablation).T

,backtest_mae,backtest fold 0,backtest fold 1,backtest fold 2,mae,rmse,r2,poisson_deviance
alle kenmerken,31.7859,27.3669,38.4969,29.4940,28.9411,53.6342,0.9309,10.6251
zonder year,37.7198,30.2382,42.3724,40.5489,35.0838,62.9455,0.9049,14.2973
zonder active_stations,32.2469,28.0590,39.1623,29.5194,29.2125,53.7854,0.9306,10.8475
zonder snow_depth_cm,31.6898,27.4778,37.8854,29.7062,30.1411,55.0788,0.9272,11.7439
zonder alle drie (= kenmerken uit 2024),74.5475,56.2783,77.6035,89.7607,78.9619,129.5677,0.5970,69.0171


**Interpretatie:**
- **`year` is onmisbaar**: zonder dat kenmerk stijgt de fout van 28,9 naar 35,1 (test) en van 31,8 naar 37,7 (backtest). Zonder `year` moet het model 13 jaar groei uitmiddelen.
- **Zonder alle drie** (de kenmerken uit 2024) is het model onbruikbaar op alle data: MAE 79, R² 0,60. Precies het probleem van de baseline "alle jaren" in `07`: het voorspelt een gemiddelde over 13 jaar.
- **`active_stations`** voegt naast `year` weinig toe (test 29,2 → 28,9, backtest 32,2 → 31,8), maar wel consistent, in elke fold. Het helpt vooral waar zones groeien.
- **`snow_depth_cm`** maakt het verschil op de testset (30,1 → 28,9) maar **niet in de backtest** (31,7 zonder, 31,8 met; in fold 1 zelfs iets slechter). De verklaring: de validatiejaren 2022–2025 hadden zachte winters met weinig sneeuw op de grond, de testwinter (2025–2026) was koud en sneeuwrijk. Het kenmerk helpt dus alleen wanneer er sneeuw ligt, en dan veel (zie `09`: de fout op 26–31 januari 2026 daalt sterk). We houden het, met die kanttekening: het effect is gemeten op één sneeuwrijke winter.

## 6. Het model voor de API
De gekozen hyperparameters gaan naar `models/best_params.json`. `train.py` hertraint het model op **alle** uren (ook de laatste 12 maanden), meet het eerst eerlijk op de testperiode en schrijft het alleen weg als het niet slechter is dan het model dat er staat (`07` zette er een eerste versie).

In [7]:
(cb.MODEL_DIR / "best_params.json").write_text(json.dumps(
    {"source": f"08_hgb_all_data ({chosen}, backtest 3 folds)", "params": params}, indent=2))
result = subprocess.run([sys.executable, "train.py"], capture_output=True, text=True)
print(result.stdout[-1500:], result.stderr[-1500:])

Data: 2,818,992 zone-uren (2,556,912 train / 262,080 test: 2025-10-01 t/m 2026-09-30), 30 zones
Hold-out: {'mae': 28.9411, 'rmse': 53.6342, 'r2': 0.9309, 'poisson_deviance': 10.6251}
Huidig model: MAE 29.0563; nieuw: 28.9411 (grens 29.6374)
Bundle geschreven naar C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI-Challenge\deploy\citi_bike_demand\model.pkl (7.1 MB)
 


## Samenvatting
| | Backtest-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| baseline (zone × uur × weekend, vorig jaar, `07`) | 57,6 | 71,0 | 0,68 |
| HistGB, hyperparameters uit 2024 | 32,0 | | |
| **HistGB, opnieuw getuned (gedeployd)** | **31,8** | **28,9** | **0,931** |
| idem, getraind op alleen het laatste jaar | | 30,3 | 0,925 |

- Het model in de API is getraind op **alle 13 jaar** (na de meting hertraind op alle uren, ook de testperiode), 7,1 MB groot.
- Meer data helpt, maar met afnemende winst; de trend-kenmerken zijn daarvoor noodzakelijk.
- De volledige vergelijking en de foutenanalyse staan in `09_model_comparison`.